# 02 - Prétraitement et Feature Engineering

## Objectifs de ce notebook
1. **Nettoyage des données** : gestion des valeurs manquantes
2. **Feature Engineering** : création de nouvelles variables pertinentes
3. **Encodage** : transformation des variables catégorielles
4. **Pipeline sklearn** : création d'un pipeline réutilisable

## Insights de l'EDA qui guident nos choix
- `Age` : 20% manquant, forte corrélation avec la survie (enfants prioritaires)
- `Cabin` : 77% manquant, mais la présence indique un statut élevé
- `Sex` : variable la plus prédictive ("women and children first")
- `Pclass` : la classe sociale impacte fortement la survie
- `Fare` : distribution asymétrique avec outliers

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import sys
import re
import warnings
warnings.filterwarnings('ignore')

# Ajouter le dossier src au path
sys.path.append('../src')

# Configuration
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Chargement des données

In [ ]:
# Charger les données brutes
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

print(f"Train: {train_df.shape[0]} lignes, {train_df.shape[1]} colonnes")
print(f"Test: {test_df.shape[0]} lignes, {test_df.shape[1]} colonnes")

# Aperçu
train_df.head()

---
## 2. Analyse des valeurs manquantes

Avant de traiter les données, identifions précisément les valeurs manquantes.

In [ ]:
def missing_values_table(df):
    """Crée un tableau récapitulatif des valeurs manquantes."""
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table.columns = ['Manquantes', '% du Total']
    mis_val_table = mis_val_table[mis_val_table['Manquantes'] > 0]
    return mis_val_table.sort_values('% du Total', ascending=False)

print("=" * 50)
print("VALEURS MANQUANTES - TRAIN")
print("=" * 50)
display(missing_values_table(train_df))

print("\n" + "=" * 50)
print("VALEURS MANQUANTES - TEST")
print("=" * 50)
display(missing_values_table(test_df))

In [ ]:
# Visualisation des valeurs manquantes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train
missing_train = train_df.isnull().sum()[train_df.isnull().sum() > 0]
axes[0].barh(missing_train.index, missing_train.values, color='coral')
axes[0].set_xlabel('Nombre de valeurs manquantes')
axes[0].set_title('Valeurs manquantes - Train')
for i, v in enumerate(missing_train.values):
    axes[0].text(v + 5, i, f'{v} ({v/len(train_df)*100:.1f}%)')

# Test
missing_test = test_df.isnull().sum()[test_df.isnull().sum() > 0]
axes[1].barh(missing_test.index, missing_test.values, color='steelblue')
axes[1].set_xlabel('Nombre de valeurs manquantes')
axes[1].set_title('Valeurs manquantes - Test')
for i, v in enumerate(missing_test.values):
    axes[1].text(v + 5, i, f'{v} ({v/len(test_df)*100:.1f}%)')

plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

### Résumé des valeurs manquantes

| Variable | Train | Test | Stratégie |
|----------|-------|------|----------|
| **Cabin** | 687 (77%) | 327 (78%) | Extraire le pont (deck) + indicateur HasCabin |
| **Age** | 177 (20%) | 86 (21%) | Imputation par médiane selon le titre |
| **Embarked** | 2 (0.2%) | 0 | Imputation par le mode ('S') |
| **Fare** | 0 | 1 | Imputation par médiane selon la classe |

---

## 3. Feature Engineering

Nous allons créer de nouvelles variables qui capturent des informations importantes pour prédire la survie.

### 3.1 Extraction du titre (depuis Name)

**Justification** : Le titre reflète le statut social, le sexe et l'âge approximatif du passager.

In [ ]:
def extract_title(name):
    """
    Extrait le titre du nom d'un passager.
    Ex: 'Braund, Mr. Owen Harris' -> 'Mr'
    """
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)
    return 'Unknown'

# Appliquer à train et test
train_df['Title'] = train_df['Name'].apply(extract_title)
test_df['Title'] = test_df['Name'].apply(extract_title)

# Voir les titres trouvés
print("Titres trouvés (Train):")
print(train_df['Title'].value_counts())

In [ ]:
# Mapping pour regrouper les titres rares
TITLE_MAPPING = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',  # Jeunes garçons
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',      # Mademoiselle
    'Countess': 'Rare',
    'Ms': 'Miss',
    'Lady': 'Rare',
    'Jonkheer': 'Rare',
    'Don': 'Rare',
    'Dona': 'Rare',
    'Mme': 'Mrs',        # Madame
    'Capt': 'Rare',
    'Sir': 'Rare'
}

train_df['Title'] = train_df['Title'].map(TITLE_MAPPING)
test_df['Title'] = test_df['Title'].map(TITLE_MAPPING)

# Remplacer les valeurs inconnues par 'Rare'
train_df['Title'].fillna('Rare', inplace=True)
test_df['Title'].fillna('Rare', inplace=True)

print("Titres après regroupement:")
print(train_df['Title'].value_counts())

In [ ]:
# Visualisation: Survie par titre
fig, ax = plt.subplots(figsize=(10, 5))

survival_by_title = train_df.groupby('Title')['Survived'].mean().sort_values(ascending=False)
colors = ['#6bcb77' if x > 0.5 else '#ff6b6b' for x in survival_by_title]

bars = ax.bar(survival_by_title.index, survival_by_title.values, color=colors, edgecolor='black')
ax.axhline(y=0.5, color='gray', linestyle='--', label='50%')
ax.set_ylabel('Taux de survie')
ax.set_title('Taux de survie par titre')
ax.set_ylim(0, 1)

for bar, val in zip(bars, survival_by_title.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/survival_by_title.png', dpi=150)
plt.show()

print("\n📊 Insight: Mrs et Miss (femmes) et Master (jeunes garçons) ont les meilleurs taux de survie.")

### 3.2 Taille de la famille

**Justification** : Voyager en famille peut affecter les chances de survie (groupes qui s'entraident vs. passagers seuls).

In [ ]:
# Créer FamilySize
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1

# Indicateur IsAlone
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)
test_df['IsAlone'] = (test_df['FamilySize'] == 1).astype(int)

# Catégories de taille de famille
def categorize_family(size):
    if size == 1:
        return 'Alone'
    elif size <= 3:
        return 'Small'
    elif size <= 5:
        return 'Medium'
    else:
        return 'Large'

train_df['FamilySizeGroup'] = train_df['FamilySize'].apply(categorize_family)
test_df['FamilySizeGroup'] = test_df['FamilySize'].apply(categorize_family)

print("Distribution de la taille des familles:")
print(train_df['FamilySizeGroup'].value_counts())

In [ ]:
# Visualisation: Survie par taille de famille
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survie par FamilySize exact
survival_by_size = train_df.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
ax1 = axes[0]
bars = ax1.bar(survival_by_size.index, survival_by_size['mean'], color='steelblue', edgecolor='black')
ax1.axhline(y=0.5, color='red', linestyle='--')
ax1.set_xlabel('Taille de la famille')
ax1.set_ylabel('Taux de survie')
ax1.set_title('Survie par taille de famille')

# Survie par groupe
order = ['Alone', 'Small', 'Medium', 'Large']
survival_by_group = train_df.groupby('FamilySizeGroup')['Survived'].mean().reindex(order)
colors = ['#ff6b6b', '#6bcb77', '#ffd93d', '#ff6b6b']
ax2 = axes[1]
bars = ax2.bar(order, survival_by_group.values, color=colors, edgecolor='black')
ax2.axhline(y=0.5, color='gray', linestyle='--')
ax2.set_xlabel('Groupe familial')
ax2.set_ylabel('Taux de survie')
ax2.set_title('Survie par groupe familial')

for bar, val in zip(bars, survival_by_group.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/survival_by_family.png', dpi=150)
plt.show()

print("\n📊 Insight: Les petites familles (2-3 personnes) ont le meilleur taux de survie.")
print("   Les passagers seuls et les très grandes familles survivent moins.")

### 3.3 Traitement de Cabin

**Justification** : Bien que 77% soit manquant, la présence d'un numéro de cabine documenté est un indicateur de statut social (passagers de 1ère classe).

In [ ]:
# Indicateur HasCabin
train_df['HasCabin'] = train_df['Cabin'].notna().astype(int)
test_df['HasCabin'] = test_df['Cabin'].notna().astype(int)

# Extraire le pont (deck) - première lettre
def extract_deck(cabin):
    if pd.isna(cabin):
        return 'Unknown'
    return cabin[0]

train_df['Deck'] = train_df['Cabin'].apply(extract_deck)
test_df['Deck'] = test_df['Cabin'].apply(extract_deck)

print("Distribution des ponts (Train):")
print(train_df['Deck'].value_counts())

In [ ]:
# Visualisation: Survie selon HasCabin
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# HasCabin vs Survie
survival_cabin = train_df.groupby('HasCabin')['Survived'].mean()
labels = ['Sans cabine', 'Avec cabine']
colors = ['#ff6b6b', '#6bcb77']
axes[0].bar(labels, survival_cabin.values, color=colors, edgecolor='black')
axes[0].set_ylabel('Taux de survie')
axes[0].set_title('Survie selon la présence de cabine')
for i, v in enumerate(survival_cabin.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# HasCabin vs Pclass (pour comprendre la corrélation)
cabin_by_class = train_df.groupby('Pclass')['HasCabin'].mean()
axes[1].bar(['1ère classe', '2ème classe', '3ème classe'], cabin_by_class.values, 
            color=['gold', 'silver', '#cd7f32'], edgecolor='black')
axes[1].set_ylabel('% avec cabine documentée')
axes[1].set_title('Présence de cabine par classe')
for i, v in enumerate(cabin_by_class.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/cabin_analysis.png', dpi=150)
plt.show()

print("\n📊 Insight: Avoir une cabine documentée corrèle avec la 1ère classe et une meilleure survie.")

---
## 4. Imputation des valeurs manquantes

### 4.1 Imputation de Age

**Stratégie choisie** : Médiane par titre

**Justification** : 
- La médiane globale (28 ans) est trop générique
- Un "Master" (jeune garçon) n'a pas le même âge qu'un "Mr" (homme adulte)
- Cette approche préserve la relation titre-âge identifiée dans l'EDA

In [ ]:
# Calculer la médiane par titre
age_by_title = train_df.groupby('Title')['Age'].median()
print("Âge médian par titre:")
print(age_by_title.sort_values())

In [ ]:
# Visualisation avant/après imputation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution avant
axes[0].hist(train_df['Age'].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(train_df['Age'].median(), color='red', linestyle='--', label=f'Médiane: {train_df["Age"].median():.1f}')
axes[0].set_xlabel('Âge')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution de Age AVANT imputation')
axes[0].legend()

# Imputation
train_df_copy = train_df.copy()
for title in train_df_copy['Title'].unique():
    mask = (train_df_copy['Age'].isnull()) & (train_df_copy['Title'] == title)
    median_age = age_by_title.get(title, train_df_copy['Age'].median())
    train_df_copy.loc[mask, 'Age'] = median_age

# Distribution après
axes[1].hist(train_df_copy['Age'], bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
axes[1].axvline(train_df_copy['Age'].median(), color='red', linestyle='--', label=f'Médiane: {train_df_copy["Age"].median():.1f}')
axes[1].set_xlabel('Âge')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution de Age APRÈS imputation')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/age_imputation.png', dpi=150)
plt.show()

print(f"\n✓ {train_df['Age'].isnull().sum()} valeurs imputées dans Age")

In [ ]:
# Appliquer l'imputation
for df in [train_df, test_df]:
    for title in df['Title'].unique():
        mask = (df['Age'].isnull()) & (df['Title'] == title)
        median_age = age_by_title.get(title, train_df['Age'].median())
        df.loc[mask, 'Age'] = median_age

print(f"Age manquant après imputation:")
print(f"  - Train: {train_df['Age'].isnull().sum()}")
print(f"  - Test: {test_df['Age'].isnull().sum()}")

### 4.2 Création des groupes d'âge

**Justification** : La catégorisation capture "women and children first" plus explicitement.

In [ ]:
def categorize_age(age):
    """Catégorise l'âge en groupes."""
    if age < 13:
        return 'Child'
    elif age < 20:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

train_df['AgeGroup'] = train_df['Age'].apply(categorize_age)
test_df['AgeGroup'] = test_df['Age'].apply(categorize_age)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))

order = ['Child', 'Teen', 'Adult', 'Senior']
survival_by_age = train_df.groupby('AgeGroup')['Survived'].mean().reindex(order)
colors = ['#6bcb77', '#ffd93d', '#ff6b6b', '#ff6b6b']

bars = ax.bar(order, survival_by_age.values, color=colors, edgecolor='black')
ax.axhline(y=0.5, color='gray', linestyle='--')
ax.set_ylabel('Taux de survie')
ax.set_title('Taux de survie par groupe d\'âge')

for bar, val in zip(bars, survival_by_age.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/survival_by_age_group.png', dpi=150)
plt.show()

print("\n📊 Insight: Les enfants (Child) ont le meilleur taux de survie (~58%), confirmant 'children first'.")

### 4.3 Imputation de Embarked

In [ ]:
# Vérifier les valeurs manquantes
print(f"Embarked manquant: {train_df['Embarked'].isnull().sum()}")
print(f"Mode de Embarked: {train_df['Embarked'].mode()[0]}")

# Imputer avec le mode
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
test_df['Embarked'].fillna(test_df['Embarked'].mode()[0], inplace=True)

print(f"\n✓ Embarked imputé avec 'S' (Southampton - port le plus fréquent)")

### 4.4 Imputation de Fare (test set seulement)

In [ ]:
# Vérifier
print(f"Fare manquant dans test: {test_df['Fare'].isnull().sum()}")

# Imputer par la médiane de la classe correspondante
if test_df['Fare'].isnull().sum() > 0:
    fare_by_class = train_df.groupby('Pclass')['Fare'].median()
    
    for pclass in test_df['Pclass'].unique():
        mask = (test_df['Fare'].isnull()) & (test_df['Pclass'] == pclass)
        test_df.loc[mask, 'Fare'] = fare_by_class[pclass]
    
    print(f"✓ Fare imputé avec la médiane par classe")
    print(fare_by_class)

---
## 5. Encodage des variables catégorielles

### 5.1 Label Encoding pour Sex (binaire)

In [ ]:
# Encoder Sex: female=0, male=1
le_sex = LabelEncoder()
train_df['Sex'] = le_sex.fit_transform(train_df['Sex'])
test_df['Sex'] = le_sex.transform(test_df['Sex'])

print("Encodage de Sex:")
print(dict(zip(le_sex.classes_, range(len(le_sex.classes_)))))

### 5.2 One-Hot Encoding pour les autres catégorielles

In [ ]:
# Colonnes à encoder en one-hot
onehot_columns = ['Embarked', 'Title', 'FamilySizeGroup', 'Deck', 'AgeGroup']

for col in onehot_columns:
    # Créer les dummies
    train_dummies = pd.get_dummies(train_df[col], prefix=col, drop_first=True)
    test_dummies = pd.get_dummies(test_df[col], prefix=col, drop_first=True)
    
    # S'assurer que train et test ont les mêmes colonnes
    for dummy_col in train_dummies.columns:
        if dummy_col not in test_dummies.columns:
            test_dummies[dummy_col] = 0
    
    # Réordonner les colonnes du test
    test_dummies = test_dummies[train_dummies.columns]
    
    # Ajouter au DataFrame
    train_df = pd.concat([train_df, train_dummies], axis=1)
    test_df = pd.concat([test_df, test_dummies], axis=1)
    
    print(f"✓ {col}: {len(train_dummies.columns)} colonnes créées")

print(f"\nNouvelles dimensions:")
print(f"  - Train: {train_df.shape}")
print(f"  - Test: {test_df.shape}")

---
## 6. Sélection des features finales

Suppression des colonnes qui ne sont plus utiles.

In [ ]:
# Colonnes à supprimer
cols_to_drop = [
    'PassengerId',  # Identifiant
    'Name',         # Texte brut (titre extrait)
    'Ticket',       # Trop de valeurs uniques, peu informatif
    'Cabin',        # Remplacé par HasCabin et Deck
    'Embarked',     # Encodé
    'Title',        # Encodé
    'FamilySizeGroup',  # Encodé
    'Deck',         # Encodé
    'AgeGroup'      # Encodé
]

# Garder PassengerId du test pour les soumissions
passenger_ids = test_df['PassengerId'].copy()

# Supprimer les colonnes
train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns], inplace=True)
test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns], inplace=True)

print("Colonnes finales:")
print(list(train_df.columns))

---
## 7. Vérification finale et sauvegarde

In [ ]:
# Vérification: plus de valeurs manquantes
print("Valeurs manquantes restantes:")
print(f"  - Train: {train_df.isnull().sum().sum()}")
print(f"  - Test: {test_df.isnull().sum().sum()}")

# Vérification: types de données
print(f"\nTypes de données (Train):")
print(train_df.dtypes.value_counts())

In [ ]:
# Aperçu final
print("=" * 60)
print("APERÇU DES DONNÉES PRÉTRAITÉES")
print("=" * 60)
train_df.head()

In [ ]:
# Sauvegarder les données prétraitées
train_df.to_csv('../data/processed/train_processed.csv', index=False)
test_df.to_csv('../data/processed/test_processed.csv', index=False)
passenger_ids.to_csv('../data/processed/passenger_ids.csv', index=False)

print("✓ Données sauvegardées dans data/processed/")
print(f"  - train_processed.csv: {train_df.shape}")
print(f"  - test_processed.csv: {test_df.shape}")

---
## 8. Récapitulatif des transformations

### Valeurs manquantes

| Variable | Stratégie | Justification |
|----------|-----------|---------------|
| Age | Médiane par titre | Préserve la relation titre-âge |
| Embarked | Mode ('S') | 2 valeurs seulement, mode raisonnable |
| Fare | Médiane par classe | Le prix dépend de la classe |
| Cabin | Non imputé | Créé HasCabin et Deck à la place |

### Features créées

| Feature | Source | Description |
|---------|--------|-------------|
| Title | Name | Titre extrait (Mr, Mrs, Miss, Master, Rare) |
| FamilySize | SibSp + Parch + 1 | Taille totale de la famille |
| IsAlone | FamilySize | 1 si voyageant seul |
| FamilySizeGroup | FamilySize | Catégorisation (Alone, Small, Medium, Large) |
| HasCabin | Cabin | 1 si cabine documentée |
| Deck | Cabin | Lettre du pont (A-G, Unknown) |
| AgeGroup | Age | Catégorisation (Child, Teen, Adult, Senior) |

### Encodage

| Variable | Méthode |
|----------|--------|
| Sex | Label Encoding (0/1) |
| Embarked, Title, FamilySizeGroup, Deck, AgeGroup | One-Hot Encoding |

### Colonnes finales

**Target**: `Survived` (0 = décédé, 1 = survécu)

**Features numériques**: Age, SibSp, Parch, Fare, FamilySize, IsAlone, HasCabin, Sex, Pclass

**Features encodées (one-hot)**: Embarked_*, Title_*, FamilySizeGroup_*, Deck_*, AgeGroup_*